# Hierarchical mergers of binary black holes

All the relevant information for the project are to be found in the pdf document present in the repo.
Note that you are assigned to project 2 (as the title said).

## Datasets 

Datasets are stored on Google Drive (link and description in the pdf document)

### Contacts

* Giuliano Iorio <giuliano.iorio@unipd.it>


## Libraries 

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import numpy as np


## Data load

Here we load the different files containing all the data that we will use for this project. We will merge all data in to one large file adding flags to the nature of the system and to the metallicitty so we don't loose information. This will make it easier in order to make an exploratory analysis

In [2]:
def read_nth_generation(path):
    # read and repair header (merge tokens that start with '(' into previous token)
    with open(path, 'r') as f:
        header = f.readline().strip()
    tokens = header.split()
    names = []
    for tok in tokens:
        if tok.startswith('(') and names:
            names[-1] = names[-1] + ' ' + tok
        else:
            names.append(tok)
    # read remaining rows using whitespace splitting and assign repaired names
    df = pd.read_csv(path, sep='\s+', header=None, names=names, skiprows=1, comment='#')
    return df

base_path = Path('fastcluster_comp_physA')
all_data = []
for filepath in base_path.glob('**/*/Dyn/*/nth_generation.txt'):
    sys_origin = filepath.parts[-4].split('_')[0]
    mettalictty = float(filepath.parts[-2])
    df = read_nth_generation(filepath)
    df['sys'] = sys_origin
    df['met'] = mettalictty
    all_data.append(df)

df_final = pd.concat(all_data, ignore_index=True, sort=False)

<positron-console-cell-2>:13: SyntaxWarning: invalid escape sequence '\s'


Now we drop the columns that have no useful information for our classification task

In [3]:
df_final.drop(columns=['c5:theta1', 'c6:theta2', 'c7:SMA(Rsun)', 'c8:ecc','c10:SMAfin(cm)', 'c11:eccfin',
       'c12:tpeters/Myr', 'c14:vkick/kms', 'c18:flag1', 'c19:flag2', 'c20:flag3', 'c21:flagSN', 'c22:flag_exch',
       'c23:flag_t3bb', 'c24:flag_evap','c26:ecc(10Hz)'], inplace=True)
df_final['c0:identifier'] = df_final['c0:identifier'].astype('category')

Now we separate the data into three different dataframes according to the type of system they belong to, this will make it easier incase we want to analyze the data in a same group

In [5]:
df_GC = df_final[df_final['sys'] == 'GC'] 
df_NSC = df_final[df_final['sys'] == 'NSC']
df_YSC = df_final[df_final['sys'] == 'YSC']

## Exploratory Data Analysis (EDA)

Now we will perform an exploration of the different parameters to detect some outliers and clean the datasets in order to make sure we have consistent measurements. 

In [6]:
df_final.columns

Index(['c0:identifier', 'c1:M1/Msun', 'c2:M2/Msun', 'c3:chi1', 'c4:chi2',
       'c9:(tDF+min(t12,t3bb))/Myr', 'c13:(ngen', '(tDF+t3bb+tpeters))/Myr',
       'c15:mrem/Msun', 'c16:arem', 'c17:vesc/kms', 'c25:Mtot/Msun',
       'c27:Ngen', 'sys', 'met'],
      dtype='object')

In [6]:
from ydata_profiling import ProfileReport

In [ ]:
profile_GC = ProfileReport(df_GC, title="Profiling Report for GC", explorative=True)
profile_GC.to_file("GC_report.html")
profile_YSC = ProfileReport(df_YSC, title="Profiling Report for YSC", explorative=True)
profile_YSC.to_file("YSC_report.html")
profile_NSC = ProfileReport(df_NSC, title="Profiling Report for NSC", explorative=True)
profile_NSC.to_file("NSC_report.html")
#profile_total = ProfileReport(df_final, title="Profiling Report for total dataset", explorative=True)
#profile_total.to_file("total_report.html")

/home/phuniverse/Desktop/master/LCP/Module_A/final_project/LCP_projects_Y8/.venv/lib/python3.13/site-packages/ydata_profiling/utils/dataframe.py:137: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.rename(columns={"index": "df_index"}, inplace=True)
Export report to file: 100%|██████████| 1/1 [00:00<00:00, 33.43it/s]


In [ ]:
df_final_no_id = df_final.drop(columns=['c0:identifier'])
numeric_cols = df_final_no_id.select_dtypes(include=[np.number]).columns.tolist()

g = sns.pairplot(df_final_no_id, 
                 vars=numeric_cols, # Only plot these
                 hue="sys", 
                 diag_kind="hist", 
                 palette="Set2")
g.map_lower(sns.kdeplot, levels=4, color=".2")

## Classification

## Results